In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
!unzip -q "/content/drive/MyDrive/CV_Project/archive.zip" -d "/content/dataset"

In [9]:
import os
import random
import shutil
import zipfile
from pathlib import Path
from sklearn.model_selection import train_test_split

drive_backup_path = Path("/content/drive/MyDrive/CV_Project/cv_project_dataset_1600.zip")
SPLIT_BASE = Path("/content/cv-project/data")

if drive_backup_path.exists():
    print(f"Archive found at {drive_backup_path}. Bypassing sampling protocol.")
    print(f"Extracting existing dataset to {SPLIT_BASE}...")

    SPLIT_BASE.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(drive_backup_path, 'r') as zip_ref:
        zip_ref.extractall(SPLIT_BASE)

    print("Extraction complete. Dataset ready for processing.")

else:
    print("Archive not found in Drive. Initiating fresh data sampling protocol...")

    base_search = Path("/content/dataset")
    found_dirs = list(base_search.rglob("train_val_call"))

    if not found_dirs:
        raise FileNotFoundError("System Error: Could not locate 'train_val_call'. Please verify the dataset structure.")

    SOURCE_BASE = found_dirs[0].parent
    print(f"Origin dataset localized successfully at: {SOURCE_BASE}")

    TARGET_CLASSES = [
        "train_val_call",
        "train_val_dislike",
        "train_val_like",
        "train_val_mute",
        "train_val_ok",
        "train_val_palm",
        "train_val_peace",
        "train_val_stop"
    ]

    for split in ["train", "val", "test"]:
        for folder_name in TARGET_CLASSES:
            label = folder_name.replace("train_val_", "")
            (SPLIT_BASE / split / label).mkdir(parents=True, exist_ok=True)

    random.seed(42)

    print("\nTarget: 200 samples/class | Distribution: 70% Train, 15% Validation, 15% Test\n")

    for folder_name in TARGET_CLASSES:
        label = folder_name.replace("train_val_", "")
        src_dir = SOURCE_BASE / folder_name

        # Grab all valid images
        images = [f for f in src_dir.iterdir() if f.is_file() and f.suffix.lower() in {".jpg", ".jpeg", ".png"}]
        selected_200 = random.sample(images, min(200, len(images)))

        # 70% Train (140), 15% Val (30), 15% Test (30)
        train_imgs, temp_imgs = train_test_split(selected_200, test_size=0.30, random_state=42)
        val_imgs, test_imgs = train_test_split(temp_imgs, test_size=0.50, random_state=42)

        for f in train_imgs:
            shutil.copy(f, SPLIT_BASE / "train" / label / f.name)
        for f in val_imgs:
            shutil.copy(f, SPLIT_BASE / "val" / label / f.name)
        for f in test_imgs:
            shutil.copy(f, SPLIT_BASE / "test" / label / f.name)

        print(f" [Processed: {label.upper():8s}] Allocated -> Train: {len(train_imgs)} | Val: {len(val_imgs)} | Test: {len(test_imgs)}")

    print(f"\n [SYSTEM] Dataset pipeline complete. Master matrix built at: {SPLIT_BASE}")

    print("\n[SYSTEM] Compressing assets for cloud transit...")
    zip_path = "/content/cv_project_dataset_1600"
    shutil.make_archive(zip_path, "zip", SPLIT_BASE)
    print(f"Archive generated successfully: {zip_path}.zip")

    if Path("/content/drive/MyDrive/CV_Project").exists():
        shutil.copy(f"{zip_path}.zip", drive_backup_path)
        print(f"Secure sync to Google Drive verified. Backup anchored at: {drive_backup_path}")

Archive found at /content/drive/MyDrive/CV_Project/cv_project_dataset_1600.zip. Bypassing sampling protocol.
Extracting existing dataset to /content/cv-project/data...
Extraction complete. Dataset ready for processing.


Preprocessing and Training

In [14]:
import json
import cv2
import numpy as np
from pathlib import Path
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score, f1_score
import joblib

In [15]:
SPLIT_BASE = Path("/content/cv-project/data")

ann_dirs = list(Path("/content/dataset").rglob("ann_train_val"))
if not ann_dirs:
    raise FileNotFoundError("Could not find 'ann_train_val' in /content/dataset. Make sure it was inside your zip!")

ANN_DIR = ann_dirs[0]
print(f"Located HaGRID JSON annotations at: {ANN_DIR}")

Located HaGRID JSON annotations at: /content/dataset/hagrid-sample-30k-384p/ann_train_val


In [16]:
TARGET_LABELS = ["call", "dislike", "like", "mute", "ok", "palm", "peace", "stop"]
BBOX_LOOKUP = {}

for label in TARGET_LABELS:
    json_path = ANN_DIR / f"{label}.json"
    if json_path.exists():
        with open(json_path, "r") as f:
            data = json.load(f)
            for img_id, info in data.items():
                bboxes = info.get("bboxes", [])
                labels = info.get("labels", [])
                # Find the bounding box that matches the target gesture label
                for box, box_label in zip(bboxes, labels):
                    if box_label == label:
                        BBOX_LOOKUP[img_id] = box
                        break
                # Fallback to first bbox if not matched
                if img_id not in BBOX_LOOKUP and bboxes:
                    BBOX_LOOKUP[img_id] = bboxes[0]

print(f"Indexed {len(BBOX_LOOKUP)} hand bounding boxes across 8 classes.")

def crop_to_hand_roi(image_bgr, img_stem):
    if img_stem not in BBOX_LOOKUP:
        return image_bgr

    img_h, img_w = image_bgr.shape[:2]
    nx, ny, nw, nh = BBOX_LOOKUP[img_stem]

    # Convert normalized [x, y, w, h] to pixel coordinates with 10% context padding
    pad_x = nw * 0.10
    pad_y = nh * 0.10

    x1 = max(0, int((nx - pad_x) * img_w))
    y1 = max(0, int((ny - pad_y) * img_h))
    x2 = min(img_w, int((nx + nw + pad_x) * img_w))
    y2 = min(img_h, int((ny + nh + pad_y) * img_h))

    crop = image_bgr[y1:y2, x1:x2]
    return crop if crop.size > 0 else image_bgr

def extract_hand_features(image_bgr, img_stem=None):
    """
    Project Outline Classical Baseline Pipeline on Hand ROI:
    1. Crop to Hand ROI (if bbox available) & Resize to 128x128
    2. 5x5 Gaussian Blur Denoising
    3. BGR -> YCrCb Skin Mask + Morphological Closing
    4. Largest Hand Contour -> 7 Hu Moments + Convexity Defects + Shape Ratios
    5. CLAHE + HOG Orientation Features
    """
    if image_bgr is None:
        return None

    # 1. Isolate the hand ROI so faces/walls don't corrupt the skin mask
    if img_stem is not None:
        image_bgr = crop_to_hand_roi(image_bgr, img_stem)

    img = cv2.resize(image_bgr, (128, 128))
    blurred = cv2.GaussianBlur(img, (5, 5), 0)

    # 2. Convert to YCrCb and threshold skin pixels
    ycrcb = cv2.cvtColor(blurred, cv2.COLOR_BGR2YCrCb)
    lower_skin = np.array([0, 133, 77], dtype=np.uint8)
    upper_skin = np.array([255, 173, 127], dtype=np.uint8)
    mask = cv2.inRange(ycrcb, lower_skin, upper_skin)

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    # 3. Extract Largest Hand Contour + 7 Hu Moments + Convexity Defects
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    hu_features = [0.0] * 7
    defect_count = 0.0
    aspect_ratio = 1.0
    solidity = 0.0

    if contours:
        largest_contour = max(contours, key=cv2.contourArea)
        area = cv2.contourArea(largest_contour)
        if area > 100:
            moments = cv2.moments(largest_contour)
            hu_moments = cv2.HuMoments(moments).flatten()
            hu_features = [
                -1.0 * np.sign(h) * np.log10(abs(h) + 1e-10) if h != 0 else 0.0
                for h in hu_moments
            ]
            x, y, w, h = cv2.boundingRect(largest_contour)
            aspect_ratio = float(w) / float(h) if h > 0 else 1.0

            hull_pts = cv2.convexHull(largest_contour)
            hull_area = cv2.contourArea(hull_pts)
            solidity = float(area) / float(hull_area) if hull_area > 0 else 0.0

            hull_idx = cv2.convexHull(largest_contour, returnPoints=False)
            if hull_idx is not None and len(hull_idx) > 3 and len(largest_contour) > 3:
                try:
                    defects = cv2.convexityDefects(largest_contour, hull_idx)
                    if defects is not None:
                        for i in range(defects.shape[0]):
                            if defects[i, 0][3] > 1000:
                                defect_count += 1.0
                except Exception:
                    defect_count = 0.0

    # 4. CLAHE Contrast Enhancement + HOG Descriptor on the cropped hand
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    gray_enhanced = clahe.apply(gray)

    hog = cv2.HOGDescriptor((128, 128), (32, 32), (16, 16), (16, 16), 9)
    hog_features = hog.compute(gray_enhanced).flatten()

    return np.hstack([hu_features, [defect_count, aspect_ratio, solidity], hog_features])

Indexed 226013 hand bounding boxes across 8 classes.


In [17]:
def load_split_data(split_name, augment=False):
    X, y = [], []
    split_dir = SPLIT_BASE / split_name
    for class_dir in sorted(split_dir.iterdir()):
        if not class_dir.is_dir():
            continue
        label = class_dir.name
        for img_path in class_dir.glob("*.*"):
            img = cv2.imread(str(img_path))
            if img is None:
                continue

            # Crop to hand bbox first, then extract features
            hand_roi = crop_to_hand_roi(img, img_path.stem)
            feats = extract_hand_features(hand_roi, img_stem=None)
            if feats is not None:
                X.append(feats)
                y.append(label)

            if augment:
                flipped_roi = cv2.flip(hand_roi, 1)
                feats_flip = extract_hand_features(flipped_roi, img_stem=None)
                if feats_flip is not None:
                    X.append(feats_flip)
                    y.append(label)

    return np.array(X), np.array(y)

print("Extracting ROI-Cropped Hand Features (Train + Flip Augmentation)...")
X_train, y_train = load_split_data("train", augment=True)

print("Extracting ROI-Cropped Hand Features (Validation)...")
X_val, y_val = load_split_data("val", augment=False)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

print("Training RBF Support Vector Machine on Hand ROIs...")
svm_model = SVC(kernel="rbf", C=10.0, gamma="scale", probability=True, random_state=42)
svm_model.fit(X_train_scaled, y_train)

y_val_pred = svm_model.predict(X_val_scaled)
val_acc = accuracy_score(y_val, y_val_pred)
val_macro_f1 = f1_score(y_val, y_val_pred, average="macro")


print("\n" + "=" * 60)
print("MILESTONE 2 HAND-ROI BASELINE RESULTS (Validation Split)")
print(f"Validation Accuracy: {val_acc * 100:.2f}%")
print(f"Validation Macro-F1: {val_macro_f1:.4f}")
print("=" * 60)
print("\nClassification Report:\n")
print(classification_report(y_val, y_val_pred))

Extracting ROI-Cropped Hand Features (Train + Flip Augmentation)...
Extracting ROI-Cropped Hand Features (Validation)...
Training RBF Support Vector Machine on Hand ROIs...

MILESTONE 2 HAND-ROI BASELINE RESULTS (Validation Split)
Validation Accuracy: 96.25%
Validation Macro-F1: 0.9627

Classification Report:

              precision    recall  f1-score   support

        call       0.88      0.97      0.92        30
     dislike       0.97      0.97      0.97        30
        like       1.00      0.90      0.95        30
        mute       1.00      1.00      1.00        30
          ok       1.00      0.97      0.98        30
        palm       1.00      0.93      0.97        30
       peace       0.94      1.00      0.97        30
        stop       0.94      0.97      0.95        30

    accuracy                           0.96       240
   macro avg       0.96      0.96      0.96       240
weighted avg       0.96      0.96      0.96       240



In [20]:
import joblib
from pathlib import Path

drive_model_dir = Path("/content/drive/MyDrive/CV_Project/models")

drive_model_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(svm_model, drive_model_dir / "svm_baseline_model.pkl")
joblib.dump(scaler, drive_model_dir / "scaler.pkl")

print(f"Models successfully saved to Google Drive at: {drive_model_dir}")

Models successfully saved to Google Drive at: /content/drive/MyDrive/CV_Project/models
